# 실습 2 · 속성 가감 — "이 게임의 PS4판"

장표 19(가감). 의미 속성이 방향으로 담기면 **벡터에 방향을 더하고 빼서 속성을 바꿀 수 있다.** 쇼핑몰 상품으로 잰다.

- 과제 : 스위치판 게임 벡터 + (PS4 방향 − 스위치 방향) → 가장 가까운 상품이 **같은 게임의 PS4판** 인가
- 정답 : 제목에서 플랫폼 이름 등을 지웠을 때 같은 게임
- 이 노트북만으로 끝난다 : 여러 플랫폼판이 있는 게임 전부 + 플랫폼마다 다른 게임 `PER_PLATFORM` 개만 임베딩(기본 2,065개 · 강사 PC 단독 약 13분, 실습 1 을 먼저 했다면 겹치는 상품은 캐시에서)
- 비교 : 가감 없이 이웃 찾기 · 메타데이터 필터(플랫폼 = PS4 인 상품만)

| 셀 | 하는 일 |
|---|---|
| 1 | 설정 |
| 2 | 준비 — DB · 임베딩 서버 · 도구 (01~04 공통) |
| 3 | 테이블 만들기 — `lab3.game_items` |
| 4 | 받기 · 표 — 같은 게임의 플랫폼판 찾기 |
| 5 | 적재 |
| 6 | 임베딩 |
| 7 | 불러오기 · 플랫폼판 쌍 |
| 8 | 플랫폼 방향 = 플랫폼별 평균의 차 (SQL `avg(embedding) … GROUP BY platform`) |
| 9 | 가감 SQL — 한 게임 : `embedding + (B 평균 − A 평균)` |
| 10 | 속성 바꾸기 성공률 — 가감 · 필터 · 둘 다 |
| 11 | 방향을 "글"로 만들면 — 플랫폼 이름 임베딩의 차 |
| 12 | 실제 임베딩은 한쪽으로 쏠려 있다 — 무작위 쌍 코사인 · 공통 성분 제거 |
| 13 | 정리 |

### 셀 1 · 설정
- `PER_PLATFORM` : 플랫폼마다 더 넣을 다른 게임 수(평균 방향의 재료이자 헷갈리게 하는 후보). `None` = 그 플랫폼 상품 전부(1만 2천여 개 · 임베딩 수십 분)

In [ ]:
# 셀 1 · 설정
DB_URL       = "postgresql://lab:lab@db:5432/lab"       # 4-1 도커의 DB
LMSTUDIO_URL = "http://host.docker.internal:1234/v1"    # LM Studio 서버 주소 (4-1 실습1 셀 1 과 같게)
EMBED_MODEL  = "text-embedding-qwen3-embedding-8b"      # LM Studio 에 로드된 임베딩 모델 이름
CACHE        = "/cache/hf/lab3"                         # 받은 원본 저장 위치 = 도커 볼륨 hfcache
BATCH_SIZE   = 16                                       # 한 번에 LM Studio 로 보내는 글 수
MAX_CHARS    = 1500                                     # 이보다 긴 글은 잘라서 넣는다
SEED         = 42
RESET        = False    # True : 이 실습의 테이블을 지우고 다시 만든다(표본 크기를 바꿨을 때). 이미 계산한 벡터는 캐시에서 돌아온다

PLATFORMS    = ["PlayStation 4", "Xbox One", "Nintendo Switch", "PC", "PlayStation 5"]
PER_PLATFORM = 300       # 플랫폼마다 다른 게임 수 (None = 전부)

### 셀 2 · 준비 — DB · 임베딩 서버 · 도구
01~04 실습의 셀 2 는 모두 같다(각 실습을 따로 돌릴 수 있게 노트북마다 들어 있다).
- DB 연결(4-1 과 같은 pgvector DB, 스키마 `lab3`) · LM Studio 확인
- **임베딩 캐시** `lab3.emb_cache` : 키 = (모델 이름, 글의 md5) → 벡터. 같은 글은 한 번만 계산한다(예 : 실습 3 에서 계산한 판례는 실습 4 에서 다시 계산하지 않음). 실습 테이블을 지워도 남고, 지우려면 `TRUNCATE lab3.emb_cache`
- `embed_table(테이블)` : 4-1 실습1 셀 7 과 같은 배치 임베딩. ① 캐시에서 먼저 채우고 ② 남은 행을 `BATCH_SIZE` 개씩 LM Studio 로 → `UPDATE` + 캐시 저장. 멈췄다 다시 실행하면 빈 행부터
- `load(SQL)` : 마지막 열이 벡터인 SELECT → (표, 벡터 행렬 · 길이 1)
- `ci(값들)` : 단위별 차이의 평균과 부트스트랩 95% 신뢰구간 — 표본이 작아 차이가 우연인지 함께 본다(구간이 0 을 포함하면 "차이가 있다고 말할 수 없음")

In [ ]:
# 셀 2 · 준비
import json, time, urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import psycopg
import requests
from pgvector.psycopg import register_vector

plt.rcParams.update({"font.family": "NanumGothic", "axes.unicode_minus": False, "figure.dpi": 110})
pd.set_option("display.max_colwidth", 70)
C = Path(CACHE); (C / "raw").mkdir(parents=True, exist_ok=True)
HF = "https://huggingface.co/datasets"
rng = np.random.default_rng(SEED)

conn = psycopg.connect(DB_URL, autocommit=True)          # 문장마다 바로 반영 (여러 문장을 묶을 때만 conn.transaction())
register_vector(conn)
conn.execute("CREATE SCHEMA IF NOT EXISTS lab3")
conn.execute("""
    CREATE TABLE IF NOT EXISTS lab3.emb_cache (          -- 임베딩 캐시 (01~04 공용)
        model     text NOT NULL,                         -- 임베딩 모델 이름 (모델이 다르면 벡터도 다르다)
        text_md5  text NOT NULL,                         -- md5(글)
        embedding vector(4096) NOT NULL,
        PRIMARY KEY (model, text_md5))""")


def one(q, params=None):
    """SQL 결과의 첫 행 첫 값"""
    row = conn.execute(q, params).fetchone()
    return row[0] if row else None


def sql(q, params=None):
    """SQL 결과 → 표"""
    with conn.cursor() as cur:
        cur.execute(q, params)
        return pd.DataFrame(cur.fetchall(), columns=[c.name for c in cur.description])


def unit(v):
    return v / np.linalg.norm(v, axis=-1, keepdims=True)


def ci(x, n=1000):
    """단위(사용자 · 질의 · 글 …)별 값 x 의 평균과 부트스트랩 95% 신뢰구간. 구간이 0 을 포함하지 않으면 ✔ (차이가 우연이라고 보기 어려움)"""
    x = np.asarray(x, float); r = np.random.default_rng(SEED)
    b = [x[r.integers(0, len(x), len(x))].mean() for _ in range(n)]
    lo, hi = np.quantile(b, [.025, .975])
    return f"{x.mean():+.3f} [{lo:+.3f}, {hi:+.3f}]" + (" ✔" if lo > 0 or hi < 0 else "")


def load(q, params=None):
    """마지막 열이 벡터인 SELECT → (표, 벡터 행렬 float32 · 길이 1). 이진 전송이라 벡터가 바로 numpy 로 온다"""
    with conn.cursor(binary=True) as cur:
        cur.execute(q, params)
        rows = cur.fetchall()
        names = [c.name for c in cur.description]
    assert rows and all(r[-1] is not None for r in rows), "임베딩이 빈 행이 있다 → 셀 6(임베딩)을 끝까지"
    return pd.DataFrame([r[:-1] for r in rows], columns=names[:-1]), unit(np.stack([r[-1] for r in rows]).astype(np.float32))


def download(url, name):
    """볼륨에 없을 때만 받는다(.part 로 받고 끝나면 이름 변경)"""
    p = C / "raw" / name
    if not p.exists():
        t = time.time(); tmp = p.with_suffix(p.suffix + ".part")
        urllib.request.urlretrieve(url, tmp); tmp.rename(p)
        print(f"  받음 {name} {p.stat().st_size / 1e6:.1f}MB · {time.time() - t:.0f}s")
    return p


def parquet_urls(ds, config, split):
    r = requests.get(f"https://huggingface.co/api/datasets/{ds}/parquet/{config}/{split}", timeout=60)
    r.raise_for_status(); return r.json()


def all_parquet_urls(ds, config="default"):
    """그 구성의 모든 분할(train · test …) parquet 주소"""
    r = requests.get(f"https://huggingface.co/api/datasets/{ds}/parquet", timeout=60); r.raise_for_status()
    return [u for us in r.json()[config].values() for u in us]


def clean(d):
    """글 다듬기 : NUL 제거 · 앞뒤 공백 · MAX_CHARS 에서 자르기"""
    d = d.reset_index(drop=True)
    d["text"] = d["text"].astype(str).str.replace("\x00", "", regex=False).str.strip().str.slice(0, MAX_CHARS)
    d["label"] = d["label"].astype(str)
    return d


def per_label(d, k):
    """라벨마다 k 개씩 무작위. 같은 SEED 면 작은 k 의 표본은 큰 k 표본의 앞부분이다(→ 캐시가 그대로 맞는다)"""
    return pd.concat([g.sample(min(len(g), k), random_state=SEED) for _, g in d.groupby("label")]).reset_index(drop=True)


def embed(texts):
    """LM Studio /v1/embeddings. 실패하면 5 · 10 · 20초 뒤 다시"""
    for wait in [5, 10, 20, None]:
        try:
            r = requests.post(f"{LMSTUDIO_URL}/embeddings", json={"model": EMBED_MODEL, "input": texts}, timeout=600)
            r.raise_for_status()
            return [np.asarray(d["embedding"], dtype=np.float32) for d in sorted(r.json()["data"], key=lambda d: d["index"])]
        except Exception as e:
            if wait is None: raise
            print(f"  ! 임베딩 실패({type(e).__name__}) → {wait}s 뒤 다시"); time.sleep(wait)


def embed_table(table):
    """빈 embedding 채우기 : ① 캐시(같은 모델 · 같은 글) → ② 남은 행을 BATCH_SIZE 개씩 LM Studio 로 · UPDATE + 캐시 저장"""
    t0 = time.time()
    hit = conn.execute(f"""UPDATE {table} t SET embedding = c.embedding FROM lab3.emb_cache c
                           WHERE t.embedding IS NULL AND c.model = %s AND c.text_md5 = md5(t.text)""", (EMBED_MODEL,)).rowcount
    todo = one(f"SELECT count(*) FROM {table} WHERE embedding IS NULL")
    print(f"[{table}] 캐시에서 {hit:,}건 · 새로 임베딩 {todo:,}건", flush=True)
    done, nb, t1 = 0, 0, time.time()
    while True:
        rows = conn.execute(f"SELECT id, text FROM {table} WHERE embedding IS NULL ORDER BY id LIMIT %s", (BATCH_SIZE,)).fetchall()
        if not rows:
            break
        vecs = embed([t for _, t in rows])
        with conn.transaction(), conn.cursor() as cur:
            cur.executemany(f"UPDATE {table} SET embedding = %s WHERE id = %s", [(v, i) for (i, _), v in zip(rows, vecs)])
            cur.executemany("INSERT INTO lab3.emb_cache VALUES (%s, md5(%s), %s) ON CONFLICT DO NOTHING",
                            [(EMBED_MODEL, t, v) for (_, t), v in zip(rows, vecs)])
        done += len(rows); nb += 1
        if nb % 25 == 0 or done >= todo:
            el = time.time() - t1; rate = done / el
            print(f"  {done:6,}/{todo:,}  {rate:5.1f}건/s  경과 {el / 60:5.1f}분  남은 시간 약 {max(todo - done, 0) / rate / 60:5.1f}분", flush=True)
    print(f"[{table}] 끝 · 캐시 {hit:,}건 + 새로 {done:,}건 · {time.time() - t0:.0f}s")


try:
    models = [m["id"] for m in requests.get(f"{LMSTUDIO_URL}/models", timeout=10).json()["data"]]
except requests.exceptions.RequestException:
    raise ConnectionError(f"LM Studio 에 연결할 수 없음 : {LMSTUDIO_URL}\n"
                          "  ① LM Studio 의 Developer 탭에서 서버가 켜져 있는지 확인\n"
                          "  ② 포트가 다르면 셀 1 의 LMSTUDIO_URL 을 고치고 셀 1 부터 다시") from None
assert EMBED_MODEL in models, f"셀 1 의 EMBED_MODEL 을 이 목록의 이름으로 바꾼다 : {models}"
print("pgvector", one("SELECT extversion FROM pg_extension WHERE extname = 'vector'"), "· 임베딩", embed(["확인"])[0].size, "차원",
      f"· 캐시 {one('SELECT count(*) FROM lab3.emb_cache WHERE model = %s', (EMBED_MODEL,)):,}건")

### 셀 3 · 테이블 만들기
`game_key` = 제목에서 플랫폼 이름 · 괄호 · "edition" 등을 지운 것. 같으면 같은 게임.

In [ ]:
# 셀 3 · 테이블
if RESET:
    conn.execute("DROP TABLE IF EXISTS lab3.game_items")
conn.execute("""
    CREATE TABLE IF NOT EXISTS lab3.game_items (
        id        bigserial PRIMARY KEY,
        asin      text UNIQUE NOT NULL,          -- 아마존 상품 코드
        title     text,
        platform  text NOT NULL,                 -- 분류 2단계 = 플랫폼
        game_key  text NOT NULL,                 -- 플랫폼을 지운 제목 (같은 게임 판정)
        text      text NOT NULL,                 -- 임베딩할 글 = 제목 | 분류 | 플랫폼 | 특징 3개 (실습 1 과 같은 글)
        embedding vector(4096))""")
conn.execute("CREATE INDEX IF NOT EXISTS game_items_key ON lab3.game_items (game_key)")
sql("SELECT count(*) AS 상품 FROM lab3.game_items")

### 셀 4 · 받기 · 표
실습 1 과 같은 상품 메타(볼륨에 있으면 다시 받지 않음). 5개 플랫폼 상품 중
① 두 플랫폼 이상으로 나온 게임의 모든 판 + ② 플랫폼마다 나머지에서 `PER_PLATFORM` 개.

In [ ]:
# 셀 4 · 받기 · 표
import re

def shop():
    """Amazon Reviews 2023 · Video_Games 5-core : (상품 표, 사용자 표)
    상품 글 = 제목 | 분류 | 플랫폼 | 특징 3개 · category = 분류 2단계(대부분 플랫폼) · 사용자 = 이력(시간순, 공백 구분) + 가려 둔 마지막 구매"""
    base = f"{HF}/McAuley-Lab/Amazon-Reviews-2023/resolve/main"
    users = pd.read_csv(download(f"{base}/benchmark/5core/last_out_w_his/Video_Games.test.csv", "shop_test.csv"))
    need = set(users["parent_asin"]) | set(" ".join(users["history"].fillna("")).split())
    rows = []
    with open(download(f"{base}/raw/meta_categories/meta_Video_Games.jsonl", "shop_meta.jsonl"), encoding="utf-8") as f:
        for line in f:
            m = json.loads(line)
            if m["parent_asin"] not in need: continue
            det, cats = m.get("details") or {}, m.get("categories") or []
            feats = " ; ".join((m.get("features") or [])[:3])
            text = " | ".join(x for x in [m.get("title") or "", "category: " + " > ".join(cats),
                                          f"platform: {det.get('Platform')}" if det.get("Platform") else "", feats] if x)
            rows.append({"asin": m["parent_asin"], "text": text, "label": cats[1] if len(cats) > 1 else "?", "title": m.get("title") or ""})
    items = clean(pd.DataFrame(rows)).rename(columns={"label": "category"})
    return items, users[["user_id", "parent_asin", "history"]]


STRIP = ["playstation 4", "playstation 5", "ps4", "ps5", "nintendo switch", "switch", "xbox one", "xbox series x", "xbox", "pc",
         "windows", "standard edition", "edition", "digital code", "physical"]


def game_key(t):
    t = re.sub(r"\(.*?\)|\[.*?\]", " ", t.lower())
    for w in STRIP: t = t.replace(w, " ")
    return " ".join(re.sub(r"[^0-9a-z ]", " ", t).split())


items_all, _ = shop()
G = items_all[items_all["category"].isin(PLATFORMS)].assign(platform=lambda d: d["category"], game_key=lambda d: d["title"].map(game_key))
G = G[G["game_key"].str.len() >= 4]
multi = G.groupby("game_key")["platform"].transform("nunique") >= 2
rest = G[~multi].rename(columns={"platform": "label"})
extra = rest if PER_PLATFORM is None else per_label(rest, PER_PLATFORM)
games = pd.concat([G[multi], extra.rename(columns={"label": "platform"})]).drop_duplicates("asin").reset_index(drop=True)
print(f"5개 플랫폼 상품 {len(G):,} · 여러 판이 있는 게임 {G[multi]['game_key'].nunique()}개({multi.sum()}판) + 다른 게임 {len(extra):,} → {len(games):,}개")
games.groupby("platform").size().to_frame("상품 수")

### 셀 5 · 적재

In [ ]:
# 셀 5 · 적재
if one("SELECT count(*) FROM lab3.game_items"):
    print("이미 적재됨 → 건너뜀")
else:
    conn.execute("""INSERT INTO lab3.game_items (asin, title, platform, game_key, text)
                    SELECT * FROM unnest(%s::text[], %s::text[], %s::text[], %s::text[], %s::text[])""",
                 [games[c].tolist() for c in ["asin", "title", "platform", "game_key", "text"]])
    conn.execute("ANALYZE lab3.game_items")
sql("SELECT platform, count(*) AS 상품, count(DISTINCT game_key) AS 게임 FROM lab3.game_items GROUP BY platform ORDER BY 2 DESC")

### 셀 6 · 임베딩
실습 1 을 먼저 했다면 겹치는 상품은 캐시에서 바로 채워진다.

In [ ]:
# 셀 6 · 임베딩
embed_table("lab3.game_items")

### 셀 7 · 불러오기 · 같은 게임의 플랫폼판 쌍
같은 `game_key` 에 플랫폼이 다른 두 상품 = 방향이 있는 쌍 (A판 → B판).

In [ ]:
# 셀 7 · 불러오기 · 쌍
items, E = load("SELECT id, title, platform, game_key, embedding FROM lab3.game_items ORDER BY id")
lab = items["platform"].values
pairs = []
for k, g in items.groupby("game_key"):
    if g["platform"].nunique() < 2: continue
    for a in g.itertuples():
        for b in g.itertuples():
            if a.platform != b.platform: pairs.append((a.Index, b.Index, a.platform, b.platform))
P = pd.DataFrame(pairs, columns=["src", "dst", "from", "to"])
print(f"상품 {len(items):,} · 여러 플랫폼판이 있는 게임 {items.groupby('game_key')['platform'].nunique().gt(1).sum()} · 방향 있는 쌍 {len(P):,}")
P.groupby(["from", "to"]).size().sort_values(ascending=False).head(8).to_frame("쌍 수")

### 셀 8 · 플랫폼 방향
플랫폼마다 상품 벡터 평균 → 두 평균의 차가 "플랫폼을 바꾸는 방향". 게임 내용은 평균에서 서로 상쇄되고 플랫폼 성분이 남는다.
평균은 SQL `avg(embedding) … GROUP BY platform` 로 구한다.

In [ ]:
# 셀 8 · 플랫폼 방향
CENT = sql("SELECT platform, count(*) AS 상품, avg(embedding) AS m, vector_norm(avg(embedding)) AS m_norm FROM lab3.game_items GROUP BY platform")
cent = {p: np.asarray(m, np.float32) for p, m in zip(CENT["platform"], CENT["m"])}
display(CENT[["platform", "상품", "m_norm"]].round(3).rename(columns={"m_norm": "‖M‖"}))
D = pd.DataFrame([[float(unit(cent[a]) @ unit(cent[b])) for b in PLATFORMS] for a in PLATFORMS], index=PLATFORMS, columns=PLATFORMS).round(3)
print("플랫폼 평균끼리의 코사인 (1 에 가까울수록 평균이 비슷)")
D

### 셀 9 · 가감 SQL — 한 게임
pgvector 는 벡터끼리 `+` · `-` 를 한다. 질의 = 원래 상품 벡터 + (B 플랫폼 평균 − A 플랫폼 평균).
코사인 거리는 길이를 보지 않으므로 다시 길이 1 로 만들 필요가 없다.
- 예 : "그대로 찾으면 1위가 정답이 아니고, 가감하면 1위가 정답"인 쌍 중 첫 쌍(Nintendo Switch → PlayStation 4 가 있으면 그것). 그런 쌍과 반대인 쌍(가감 때문에 1위를 놓친 쌍)의 수를 함께 출력

In [ ]:
# 셀 9 · 가감 SQL
def swap_sql(expr):
    return f"""
    SELECT i.title AS 결과, i.platform AS 플랫폼, round((1 - (i.embedding <=> q.v))::numeric, 3) AS 코사인, i.id = %(dst)s AS 정답
    FROM lab3.game_items i,
         (SELECT {expr} AS v FROM lab3.game_items s WHERE s.id = %(src)s) q
    WHERE i.id <> %(src)s
    ORDER BY i.embedding <=> q.v
    LIMIT 5"""


DIRECTION = """((SELECT avg(embedding) FROM lab3.game_items WHERE platform = %(b)s)
             - (SELECT avg(embedding) FROM lab3.game_items WHERE platform = %(a)s))"""
def top1(q, s):
    sc = E @ unit(q); sc[s] = -np.inf; return int(sc.argmax())


won = [(top1(E[s], s) == d, top1(E[s] + cent[b] - cent[a], s) == d) for s, d, a, b in P.itertuples(index=False)]
gain = P[[not w0 and w1 for w0, w1 in won]]
print(f"쌍 {len(P)}개 중 가감으로 1위가 된 쌍 {len(gain)} · 가감 때문에 1위를 놓친 쌍 {sum(w0 and not w1 for w0, w1 in won)}")
sel = gain[(gain["from"] == "Nintendo Switch") & (gain["to"] == "PlayStation 4")]
ex = (sel if len(sel) else gain).iloc[0]
arg = {"src": int(items["id"][ex.src]), "dst": int(items["id"][ex.dst]), "a": ex["from"], "b": ex["to"]}
print("원래 상품 :", items["title"][ex.src], f"({ex['from']})  →  찾을 것 : 같은 게임의 {ex['to']}판")
for name, expr in [("그대로 : 원래 상품 벡터", "s.embedding"), ("가감 : 원래 상품 + (B 평균 − A 평균)", "s.embedding + " + DIRECTION)]:
    print(f"\n[{name}]")
    display(sql(swap_sql(expr), arg))

### 셀 10 · 속성 바꾸기 성공률
쌍 (게임 X 의 A판 → 같은 게임의 B판) 마다 질의를 만들어 테이블 전체에서 찾는다(원래 상품은 제외).

| 방식 | 질의 | 후보 |
|---|---|---|
| 그대로 | A판 벡터 | 전체 |
| 가감 | A판 + (B 평균 − A 평균) | 전체 |
| 필터 | A판 벡터 | 플랫폼 = B 인 상품만 (메타데이터) |
| 가감 + 필터 | A판 + (B − A) | 플랫폼 = B |

1위 = B판이면 성공. MRR 도 함께.

In [ ]:
# 셀 10 · 속성 바꾸기 성공률
RS = {}                                                            # 방식 → 쌍별 순위


def evaluate_swap(query, only_dst_platform, name=None):
    r = []
    for s, d, a, b in P.itertuples(index=False):
        sc = E @ unit(query(s, a, b)); sc[s] = -np.inf
        if only_dst_platform: sc[lab != b] = -np.inf
        r.append(int((sc > sc[d]).sum()) + 1)
    r = np.array(r)
    if name: RS[name] = r
    return {"1위 성공": round(float((r == 1).mean()), 3), "상위 5": round(float((r <= 5).mean()), 3), "MRR": round(float((1 / r).mean()), 3)}


def game_ci(a, b):
    """1위 성공률 차이 a − b 의 신뢰구간. 같은 게임의 쌍끼리는 독립이 아니라 게임 단위로 묶어 부트스트랩"""
    g = pd.DataFrame({"game": items["game_key"].values[P["src"].values], "d": (RS[a] == 1).astype(float) - (RS[b] == 1)})
    return ci(g.groupby("game")["d"].mean())


same = lambda s, a, b: E[s]
swap = lambda s, a, b: E[s] + (cent[b] - cent[a])
R10 = pd.DataFrame({"그대로": evaluate_swap(same, False, "그대로"), "가감": evaluate_swap(swap, False, "가감"),
                    "필터": evaluate_swap(same, True, "필터"), "가감 + 필터": evaluate_swap(swap, True, "가감 + 필터")}).T
print(f"쌍 {len(P):,} · 게임 {len(set(items['game_key'].values[P['src'].values]))} · 후보 {len(items):,}")
print("1위 성공 차이 [게임 단위 95% 구간] : 가감 − 그대로", game_ci("가감", "그대로"), "· 필터 − 가감", game_ci("필터", "가감"),
      "· 가감 + 필터 − 필터", game_ci("가감 + 필터", "필터"))
R10

### 셀 11 · 방향을 "글"로 만들면
데이터 평균 대신 플랫폼 이름 두 개를 임베딩해 그 차를 방향으로 쓴다(king − man + woman 과 같은 방식).

In [ ]:
# 셀 11 · 이름 임베딩의 차
name_vec = dict(zip(PLATFORMS, embed(PLATFORMS)))
by_name = lambda s, a, b: E[s] + (name_vec[b] - name_vec[a])
R11 = pd.DataFrame({"데이터 평균의 차 (셀 10)": evaluate_swap(swap, False), "이름 임베딩의 차": evaluate_swap(by_name, False)}).T
R11

### 셀 12 · 실제 임베딩은 한쪽으로 쏠려 있다
장표 19 : 4096차원에서 무작위 두 방향의 코사인은 거의 0(±0.03). 실제 임베딩도 그런가?
- 무작위 2,000쌍의 코사인 평균 · 표준편차 : 가우스 무작위 방향 · 이 테이블의 상품(같은 플랫폼 쌍 · 다른 플랫폼 쌍)
- **공통 성분 제거** (장표 23) : 전체 평균을 빼고 다시 길이 1 로 → 쏠림이 줄어드는지

In [ ]:
# 셀 12 · 쏠림(비등방성)과 공통 성분 제거
g = unit(rng.standard_normal((2000, 4096)).astype(np.float32))
c = (g[:1000] * g[1000:]).sum(1)
rows = [{"쌍": "무작위 방향(가우스)", "평균": c.mean(), "표준편차": c.std()}]
Xc = unit(E - E.mean(axis=0))
i, j = rng.integers(0, len(E), 4000), rng.integers(0, len(E), 4000)
keep = i != j
for name, m in [("상품 무작위 쌍", keep), ("같은 플랫폼 쌍", keep & (lab[i] == lab[j])), ("다른 플랫폼 쌍", keep & (lab[i] != lab[j]))]:
    c, cc = (E[i[m]] * E[j[m]]).sum(1), (Xc[i[m]] * Xc[j[m]]).sum(1)
    rows.append({"쌍": name, "평균": c.mean(), "표준편차": c.std(), "공통 성분 제거 후 평균": cc.mean()})
pd.DataFrame(rows).set_index("쌍").round(3)

### 셀 13 · 정리
- 셀 10 : 가감만으로 속성이 바뀌는가, 메타데이터 필터와 비교하면 · 셀 11 : 이름 두 개의 차는 데이터 평균의 차만큼 쓸 만한가
- 셀 12 : 실제 임베딩은 무작위보다 서로 훨씬 비슷(쏠림) → 가감 · 평균의 간섭이 이론보다 크다

In [ ]:
# 셀 13 · 정리
display(R10)
R11